# Group Trajectory Overlay

Normalized overlay of embedding / attention / MLP parameter-space trajectories
on shared PC axes. Each group's trajectory is independently centered and scaled
so shape and loop structure are preserved but magnitude differences are removed.

**Intersection analysis:** For each epoch, computes pairwise L2 distance between
the three groups in normalized PC space. Epochs of minimum distance (closest
approach) are candidates for phase-lock events.

In [4]:
import numpy as np
import plotly.graph_objects as go
from pathlib import Path

from miscope import load_family

In [5]:
FAMILY_NAME = "modulo_addition_1layer"
EXPORT_DIR = Path("exports/group_overlay")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Variants to run — None means all variants with parameter_trajectory artifacts
INCLUDE = None

family = load_family(FAMILY_NAME)
all_variants = family.list_variants()

candidates = [
    v for v in all_variants
    if v.artifacts.has_cross_epoch("parameter_trajectory")
    and (INCLUDE is None or v.name in INCLUDE)
]
print(f"{len(candidates)} variants with parameter_trajectory artifacts")
for v in candidates:
    print(f"  {v.name}")

30 variants with parameter_trajectory artifacts
  modulo_addition_1layer_p107_seed999_dseed598
  modulo_addition_1layer_p113_seed999_dseed999
  modulo_addition_1layer_p103_seed999_dseed598
  modulo_addition_1layer_p101_seed999_dseed999
  modulo_addition_1layer_p113_seed485_dseed598
  modulo_addition_1layer_p59_seed999_dseed598
  modulo_addition_1layer_p59_seed485_dseed999
  modulo_addition_1layer_p113_seed999_dseed42
  modulo_addition_1layer_p59_seed485_dseed42
  modulo_addition_1layer_p113_seed999_dseed598
  modulo_addition_1layer_p97_seed999_dseed598
  modulo_addition_1layer_p101_seed999_dseed598
  modulo_addition_1layer_p101_seed485_dseed598
  modulo_addition_1layer_p59_seed999_dseed999
  modulo_addition_1layer_p101_seed999_dseed42
  modulo_addition_1layer_p101_seed485_dseed999
  modulo_addition_1layer_p127_seed999_dseed598
  modulo_addition_1layer_p107_seed485_dseed598
  modulo_addition_1layer_p89_seed999_dseed598
  modulo_addition_1layer_p127_seed485_dseed598
  modulo_addition_1la

In [6]:
def normalize_trajectory(pc_x, pc_y):
    """Center and scale so the widest axis spans [-0.5, 0.5]."""
    cx, cy = pc_x.mean(), pc_y.mean()
    nx, ny = pc_x - cx, pc_y - cy
    scale = max(nx.max() - nx.min(), ny.max() - ny.min())
    if scale > 1e-12:
        nx, ny = nx / scale, ny / scale
    return nx, ny


def compute_pairwise_distances(data, col_x=0, col_y=1):
    """Compute sign-corrected pairwise L2 distance between normalized group trajectories.

    PCA sign is arbitrary — a trajectory and its mirror image are structurally
    identical. For each pair, takes min(dist(A, B), dist(A, -B)) at every epoch
    so that mirrored trajectories read as close rather than far.
    """
    groups = {}
    for name in ("embedding", "attention", "mlp"):
        proj = data[f"{name}__projections"]
        nx, ny = normalize_trajectory(proj[:, col_x], proj[:, col_y])
        groups[name] = np.stack([nx, ny], axis=1)  # (n_epochs, 2)

    def sign_corrected_dist(a, b):
        return np.minimum(
            np.linalg.norm(a - b, axis=1),
            np.linalg.norm(a + b, axis=1),
        )

    return {
        "emb_attn": sign_corrected_dist(groups["embedding"], groups["attention"]),
        "emb_mlp":  sign_corrected_dist(groups["embedding"], groups["mlp"]),
        "attn_mlp": sign_corrected_dist(groups["attention"], groups["mlp"]),
    }

In [7]:
# Render both overlay views + intersection plot for a single variant

def render_variant(variant, show=True, export=True):
    name = variant.name
    ctx = variant.at(None)

    for view_name, suffix in [
        ("parameters.pca.group_overlay",         "pc1_pc2"),
        ("parameters.pca.group_overlay_pc2_pc3", "pc2_pc3"),
    ]:
        fig = ctx.view(view_name).figure()
        if show:
            fig.show()
        if export:
            out = EXPORT_DIR / f"{name}__overlay_{suffix}.png"
            fig.write_image(str(out))
            print(f"  saved {out.name}")

    # Intersection distance plot
    data = variant.artifacts.load_cross_epoch("parameter_trajectory")
    epochs = data["epochs"].tolist()

    for pc_label, col_x, col_y in [("PC1/PC2", 0, 1), ("PC2/PC3", 1, 2)]:
        dists = compute_pairwise_distances(data, col_x=col_x, col_y=col_y)
        colors = {
            "emb_attn": "royalblue",
            "emb_mlp":  "darkorange",
            "attn_mlp": "seagreen",
        }
        labels = {
            "emb_attn": "embedding \u2194 attention",
            "emb_mlp":  "embedding \u2194 MLP",
            "attn_mlp": "attention \u2194 MLP",
        }
        fig_d = go.Figure()
        for key, label in labels.items():
            fig_d.add_trace(go.Scatter(
                x=epochs, y=dists[key].tolist(),
                mode="lines", name=label,
                line=dict(color=colors[key], width=2),
            ))
        fig_d.update_layout(
            title=f"{name} — trajectory proximity ({pc_label}, normalized)",
            xaxis_title="Epoch",
            yaxis_title="L2 distance (normalized)",
            template="plotly_white",
            height=350,
            legend=dict(orientation="h", y=1.12),
        )
        if show:
            fig_d.show()
        if export:
            suffix = pc_label.replace("/", "_").lower()
            out = EXPORT_DIR / f"{name}__proximity_{suffix}.png"
            fig_d.write_image(str(out))
            print(f"  saved {out.name}")

In [ ]:
# Single variant — quick look
variant = family.get_variant(prime=109, seed=485, data_seed=598)
render_variant(variant, show=True, export=False)

In [ ]:
# Batch export all variants (show=False for speed)
for i, v in enumerate(candidates):
    print(f"[{i+1}/{len(candidates)}] {v.name}")
    try:
        render_variant(v, show=False, export=True)
    except Exception as e:
        print(f"  FAILED: {e}")

In [ ]:
# Cross-variant summary: final Attention-MLP distance vs performance
# Tests the hypothesis that higher final Attention-MLP distance correlates with degraded generalization

import json

rows = []
for v in candidates:
    data = v.artifacts.load_cross_epoch("parameter_trajectory")

    attn = data["attention__projections"]
    mlp  = data["mlp__projections"]
    nx_a, ny_a = normalize_trajectory(attn[:, 0], attn[:, 1])
    nx_m, ny_m = normalize_trajectory(mlp[:, 0],  mlp[:, 1])
    a = np.stack([nx_a, ny_a], axis=1)
    m = np.stack([nx_m, ny_m], axis=1)
    dists = np.minimum(np.linalg.norm(a - m, axis=1), np.linalg.norm(a + m, axis=1))
    final_dist = float(dists[-1])

    summary_path = v.variant_dir / "variant_summary.json"
    final_test_loss = None
    if summary_path.exists():
        summary = json.loads(summary_path.read_text())
        final_test_loss = summary.get("test_loss_final")

    rows.append({"name": v.name, "final_attn_mlp": final_dist, "final_test_loss": final_test_loss})

rows.sort(key=lambda r: r["final_attn_mlp"])
print(f"{'variant':<55} {'final attn↔mlp':>16} {'final test loss':>16}")
print("-" * 90)
for r in rows:
    loss_str = f"{r['final_test_loss']:.4e}" if r["final_test_loss"] is not None else "N/A"
    print(f"{r['name']:<55} {r['final_attn_mlp']:>16.3f} {loss_str:>16}")

In [ ]:
# Derivative of Attention-MLP proximity (leading indicator check)
# d(attn_mlp)/dt via finite difference; peaks in derivative precede reorganization events

def render_attn_mlp_derivative(variant, col_x=0, col_y=1, show=True, export=True):
    name = variant.name
    data = variant.artifacts.load_cross_epoch("parameter_trajectory")
    epochs = data["epochs"].tolist()

    groups = {}
    for gname in ("attention", "mlp"):
        proj = data[f"{gname}__projections"]
        nx, ny = normalize_trajectory(proj[:, col_x], proj[:, col_y])
        groups[gname] = np.stack([nx, ny], axis=1)

    dist = np.minimum(
        np.linalg.norm(groups["attention"] - groups["mlp"], axis=1),
        np.linalg.norm(groups["attention"] + groups["mlp"], axis=1),
    )

    # Finite difference — epoch spacing is non-uniform, use actual gaps
    epoch_arr = np.array(epochs)
    dt = np.diff(epoch_arr)
    deriv = np.diff(dist) / dt

    from plotly.subplots import make_subplots
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                        subplot_titles=["Attention ↔ MLP distance", "d(distance)/dt"])

    fig.add_trace(go.Scatter(x=epochs, y=dist.tolist(), mode="lines",
                             line=dict(color="seagreen", width=2), showlegend=False),
                  row=1, col=1)

    mid_epochs = ((epoch_arr[:-1] + epoch_arr[1:]) / 2).tolist()
    fig.add_trace(go.Scatter(x=mid_epochs, y=deriv.tolist(), mode="lines",
                             line=dict(color="darkorchid", width=2), showlegend=False),
                  row=2, col=1)
    fig.add_hline(y=0, line_dash="dot", line_color="gray", line_width=1, row=2, col=1)

    fig.update_layout(title=f"{name} — Attention↔MLP proximity and rate of change",
                      template="plotly_white", height=450,
                      margin=dict(l=60, r=20, t=50, b=50))
    fig.update_yaxes(title_text="L2 distance", row=1, col=1)
    fig.update_yaxes(title_text="Δdistance / epoch", row=2, col=1)
    fig.update_xaxes(title_text="Epoch", row=2, col=1)

    if show:
        fig.show()
    if export:
        out = EXPORT_DIR / f"{name}__attn_mlp_deriv.png"
        fig.write_image(str(out))
        print(f"  saved {out.name}")


# Run on variants of interest
for prime, seed, dseed in [(109, 485, 598), (101, 999, 598), (89, 999, 598), (59, 485, 598)]:
    try:
        v = family.get_variant(prime=prime, seed=seed, data_seed=dseed)
        render_attn_mlp_derivative(v, show=True, export=True)
    except Exception as e:
        print(f"p{prime}/s{seed}/ds{dseed}: {e}")